# Cinema Hall Ticket Demand Forecasting
## Exploratory Data Analysis, Leakage Audit & Multi-Model Evaluation Benchmark
**Algions AI Labs Private Limited | AI & Machine Learning Track**

---

### Executive Assessment Context
This notebook provides the foundational research, exploratory data analysis, data hygiene audits, and model benchmarking for the **Cinema Ticket Demand Forecasting** decision-support system.

### Operational Decisions Targeted ($T-3$ Days):
1. **Dynamic Screen Reallocation ($T-3$ days):** Swapping movie allocations across auditoriums before public ticket sales surge.
2. **Staff Rostering & F&B Kitchen Staging ($T-1$ day):** Accurately predicting aggregate lobby footfall per 2-hour window.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 5)
print("Libraries successfully imported. Python:", sys.version)

## 1. Data Ingestion & Quality Profiling

In [ ]:
data_path = "../data/raw/cinemaTicket_Ref.csv"
df_raw = pd.read_csv(data_path)
print(f"Raw Dataset Loaded: {df_raw.shape[0]:,} screening rows across {df_raw.shape[1]} columns.")
df_raw.head()

## 2. Data Hygiene & Anomaly Cleaning
- Systematically audits for corrupted negative capacities and null values.
- Rounds fractional capacity artifacts (`capacity = tickets_sold / occu_perc`) to true physical seats.

In [ ]:
# Audit missing & negative capacity values
print("Null values per column:")
print(df_raw.isna().sum())

# Filter invalid capacities and convert date
df_clean = df_raw[(df_raw["capacity"].notna()) & (df_raw["capacity"] > 0) & (df_raw["tickets_sold"].notna())].copy()
df_clean["capacity"] = df_clean["capacity"].round().astype(int)
df_clean["tickets_sold"] = df_clean["tickets_sold"].clip(lower=0).round().astype(int)
df_clean["tickets_sold"] = np.minimum(df_clean["tickets_sold"], df_clean["capacity"])
df_clean["date"] = pd.to_datetime(df_clean["date"])
print(f"Cleaned Dataset: {len(df_clean):,} valid rows retained.")

## 3. Strict Leakage Audit ($T-3$ Horizon)
> [!WARNING]
> **Contemporaneous Target Leakage:** Columns `total_sales`, `occu_perc`, and `tickets_out` directly calculate same-day ticket revenue ($S = y \times P$). These are strictly quarantined from all models.

In [ ]:
leakage_columns = ["total_sales", "occu_perc", "tickets_out", "ticket_use"]
print("Quarantined contemporaneous columns:", leakage_columns)
df_safe = df_clean.drop(columns=leakage_columns)
print("Leakage-safe columns remaining:", list(df_safe.columns))

## 4. Chronological Train / Validation / Test Splitting
To prevent temporal lookahead leakage, data is partitioned strictly in chronological sequence:
- **Train (65%):** Feb 21, 2018 – Aug 31, 2018 (~92,000 shows)
- **Validation (15%):** Sep 01, 2018 – Sep 30, 2018 (~23,000 shows)
- **Held-Out Test (20%):** Oct 01, 2018 – Nov 04, 2018 (~27,000 shows)

In [ ]:
train_df = df_safe[df_safe["date"] < "2018-09-01"].copy()
val_df = df_safe[(df_safe["date"] >= "2018-09-01") & (df_safe["date"] < "2018-10-01")].copy()
test_df = df_safe[df_safe["date"] >= "2018-10-01"].copy()
print(f"Splits: Train = {len(train_df):,} | Val = {len(val_df):,} | Test = {len(test_df):,}")

## 5. Model Evaluation Benchmark & Cold-Start Analysis
Evaluates **Baseline (Moving Average)** vs **LightGBM GBDT** vs **PyTorch Deep TabNet** vs **Ensemble** on Held-Out Test Set.

In [ ]:
# Load serialized metrics
with open("../artifacts/metrics_summary.json", "r") as f:
    metrics = json.load(f)

rows = []
for model_name, data in metrics.items():
    rows.append({
        "Model": model_name,
        "Test MAE (Seats)": data["overall"]["mae"],
        "Test RMSE": data["overall"]["rmse"],
        "Test WAPE (%)": f"{data['overall']['wape_pct']:.2f}%",
        "Occupancy MAE (%)": f"{data['overall']['occupancy_mae_pct']:.2f}%",
        "Turnaway Lost Tickets": f"{data['overall']['under_allocation_tickets']:,}",
    })
pd.DataFrame(rows)

## 6. Live Inference Verification
Tests real model inference via FastAPI endpoints.

In [ ]:
import urllib.request
payload = {
    "film_code": 1492,
    "cinema_code": 304,
    "capacity": 200,
    "show_time": 3,
    "date": "2024-11-22",
    "ticket_price": 18.0,
    "model_choice": "ensemble",
    "screen_type": "IMAX Laser 3D",
    "marketing_tier": "Opening Weekend Blitz",
    "weather_condition": "Rainy / Bad Weather"
}
req = urllib.request.Request(
    "http://127.0.0.1:8000/predict",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"}
)
response = json.loads(urllib.request.urlopen(req).read())
print("Inference Response:")
print(f"- Unconstrained Demand: {response['unconstrained_demand']} Patrons")
print(f"- Admitted Tickets: {response['admitted_tickets']} / {response['screen_capacity']} Seats")
print(f"- Spillover Demand: {response['spillover_demand']} Patrons")
print(f"- Multi-Show Recommended: {response['multi_show_recommended']}")
print(f"- Box Office Gross: ${response['revenue_summary']['box_office_gross']:,}")
print(f"- Concessions Gross: ${response['revenue_summary']['projected_concession_revenue']:,}")
print(f"- Staffing Roster: {response['labor_roster']['ushers_count']} Ushers, {response['labor_roster']['concessions_staff']} Concessionaires")